# Réseaux de neuronnes

## Imports & Variables

In [4]:
import os
import sys
sys.path.append(os.path.abspath("../../../"))

from src.methodes import *
from src.visualisations import *
from src.data import *

from keras.models import load_model


In [5]:
valeur_de_travail = 'T_Q'
window_size = 6

racine = "../../../"

fichier_nappe = racine + "data/fusion/data_03288X0042_P.csv"
dossier_nappe = racine + "data/fusion"

dossier_model = racine + "models/"

fichier_scaler = racine + "scaler/scaler.save"

df = charger_fichier(fichier_nappe)

In [6]:
X_train, X_val, y_train, y_val, scaler = train_data(charger_dossier(dossier_nappe), window_size, fichier_scaler)

## CNN

In [ ]:
cnn(X_train, y_train, X_val, y_val, dossier_model)

## LSTM

In [ ]:
lstm(X_train, y_train, X_val, y_val, dossier_model)

## BILSTM

In [ ]:
bilstm(X_train, y_train, X_val, y_val, dossier_model)

In [ ]:
features = ["niveau_nappe_eau","lon","lat","time_num","ETP_Q","PRELIQ_Q","T_Q","surface_imp","surface_totale"]

# 2. Chargement des modèles avec la perte masquée
print("Chargement des modèles...")

df['time_num'] = df['time'].astype('int64') // 10**9

custom_objs = {'masked_mse': masked_mse}
model_lstm = load_model(dossier_model + "CNN.keras", custom_objects={'masked_mse': masked_mse})
model_bilstm = load_model(dossier_model + "BILSTM.keras", custom_objects={'masked_mse': masked_mse})
model_bilstm = load_model(dossier_model + "LSTM.keras", custom_objects={'masked_mse': masked_mse})

mon_scaler = joblib.load(fichier_scaler)
arr_lstm = lstm_predict_array(df, model_lstm, mon_scaler, features, window_size=6, target_col=valeur_de_travail)
arr_bilstm = lstm_predict_array(df, model_bilstm, mon_scaler, features, window_size=6, target_col=valeur_de_travail)

methods = {
    "LSTM": arr_lstm,
    "BILSTM": arr_bilstm,
    "Mesures": df[valeur_de_travail],
}

# 5. Graphique
plt.figure(figsize=(12,6))
for label, arr in methods.items():
    plt.plot(df['time'], arr, label=label, alpha=0.7)

plt.title(f"Comparaison LSTM vs BILSTM sur {valeur_de_travail}")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()